# MedGemma QLoRA Fine-tuning — v2 (구조화 파이프라인)

## 목적

사전학습된 LLM(기본 `google/medgemma-4b-it`, `MODEL_NAME`으로 교체 가능)에 한국어 의료상담
데이터를 SFT(Supervised Fine-Tuning)로 학습시킨다. `main_train_llm_lora.ipynb`(기존 메인,
KorMedMCQA+GenMedGPT-5k-ko 2종 조합)과는 별개 파일 — 기존 어댑터를 건드리지 않는다.

## v1(main_train_llm_lora.ipynb / alt_train_llm_lora.ipynb) 대비 달라진 점

- 여러 데이터셋을 `messages`(+ `source_dataset`/`data_type` 메타데이터) 공통 포맷으로 통일
- `DATASET_CONFIG`에서 데이터셋별 컬럼/포맷을 명시적으로 관리(자동 추정에만 의존하지 않음)
- reasoning 태그(`<think>`, `<analysis>`, `Reasoning:`, `Step N:`) 스트리핑, 개인정보(전화번호/이메일)
  마스킹, zero-width/제어문자 제거, Unicode 정규화 등 전처리를 별도 유틸리티로 분리
- 전처리 전/후 개수, 제거 사유별 개수를 실제로 로그로 출력(품질 검증 단계)
- 공식 Train/Validation/Test(80/10/10) 분리 + 분리 전 유사 질문 dedup으로 데이터 누수 방지
- Validation loss 기준 **Early Stopping**
- GPU가 Ampere 이상(A100/L4, 유료 Colab)이면 bf16, T4(무료 Colab)면 fp16으로 **자동 분기** —
  파일 하나로 무료/유료 두 환경 다 대응

## 데이터셋 (alt_train_llm_lora.ipynb와 동일한 11개)

- **medical_knowledge(7)**: Asan-AMC-Healthinfo, MedQA(ko), KoMedInstruct-52k, snuh/ClinicalQA,
  AI_healthcare_QA, HealthSearchQA-ko, medical-korean-alpaca
- **medical_reasoning(3)**: medical-o1-reasoning-SFT-Ko, ChainOfDiagnosis-Ko, MedQA-Evol-Korean
- **conversation_style(1)**: ko_medical_chat

## 실행 순서

CELL 1 GPU/환경 확인 → 2 라이브러리 설치 → **(런타임 재시작)** → 3 CONFIG → 4 HF 로그인 →
5 데이터셋 로드 → 6 구조/샘플 분석 → 7 공통 전처리 함수 → 8 컬럼 매핑/변환 → 9 품질 검증 →
10 병합/비율 확인 → 11 Train/Val/Test 분리 → 12 모델/Tokenizer 로드 → 13 Chat Template 적용 →
14 QLoRA 설정 → 15 Trainer 설정 → 16 Fine-tuning 실행 → 17 Loss 확인 → 18 Test 평가 →
19 Base vs Fine-tuned 비교 → 20 저장


---

# CELL 1. GPU 및 환경 확인

학습 가능한 GPU가 붙어있는지, 어떤 세대인지 먼저 확인한다 — 이 결과(Ampere 이상인지)에 따라
14번(QLoRA 설정)에서 bf16/fp16이 자동으로 갈린다.

In [ ]:
!nvidia-smi


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Compute Capability: {major}.{minor}", "(Ampere+)" if major >= 8 else "(Ampere 미만 — 예: T4)")
    print("GPU Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    raise RuntimeError("GPU가 연결되어 있지 않습니다. 런타임 유형을 GPU로 변경하세요.")


---

# CELL 2. 라이브러리 설치

numpy는 Colab 기본 이미지에 이미 다른 사전 설치 패키지들과 맞춰져 있어서 버전을 강제로
낮추지 않는다(과거 세션에서 `numpy==1.26.4` 강제 고정이 오히려 ABI 충돌을 냈던 이력 있음).
`torchvision`은 텍스트 전용 파인튜닝엔 불필요하고, Colab 기본 이미지의 torch/torchvision
버전 불일치로 멀티모달 모델 클래스 로드 시 에러(`torchvision::nms`)를 낼 수 있어 제거한다.

## 설치 후 반드시 "런타임 → 세션(런타임) 다시 시작" 하세요.

In [ ]:
%pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub trl
!pip uninstall -y -q torchvision
print("설치 완료 — 런타임을 다시 시작한 뒤 CELL 3부터 이어서 실행하세요.")


---

# CELL 3. CONFIG 설정

모델/데이터셋/전처리/학습 관련 값을 전부 여기 한 곳에 모은다 — 이후 셀들은 전부 이 딕셔너리를
참조만 하고, 값을 직접 하드코딩하지 않는다.

`DATASET_CONFIG`의 각 항목:

- `repo`: Hugging Face 데이터셋 repo id
- `data_type`: `medical_knowledge` / `conversation_style` / `medical_reasoning` 중 하나
- `format`: 이 데이터셋을 어떻게 `messages`로 변환할지 — `qa_pair`(질문/답변 컬럼 1쌍),
  `mcq_abcd`(MedQA류 4지선다 객관식), `dialogue`(멀티턴 대화 리스트) 중 하나
- `question_col` / `answer_col`: `qa_pair`일 때 실제 컬럼명 (자동판별에 의존하지 않고 명시)
- `sample_cap`: 소스당 샘플링 상한(`None`이면 전량 사용)
- `strip_reasoning`: `Complex_Cot`처럼 풀이과정이 섞인 데이터에서 태그/구조를 정리할지 여부

In [ ]:
MODEL_NAME = "google/medgemma-4b-it"

CONFIG = {
    "model_name": MODEL_NAME,
    "seed": 42,

    # 8번(컬럼 매핑/변환)에서 이 순서대로 순회한다.
    "dataset_config": [
        {"repo": "ChuGyouk/Asan-AMC-Healthinfo", "data_type": "medical_knowledge",
         "format": "qa_pair", "question_col": "instruction", "answer_col": "output",
         "sample_cap": 5000, "strip_reasoning": False, "hf_config": None},
        {"repo": "ChuGyouk/MedQA", "data_type": "medical_knowledge",
         "format": "mcq_abcd", "question_col": "question", "answer_col": "answer",
         "sample_cap": 5000, "strip_reasoning": False, "hf_config": "ko"},
        {"repo": "ChuGyouk/KoMedInstruct-52k", "data_type": "medical_knowledge",
         "format": "qa_pair", "question_col": "instruction", "answer_col": "output",
         "sample_cap": 5000, "strip_reasoning": False, "hf_config": None},
        {"repo": "snuh/ClinicalQA", "data_type": "medical_knowledge",
         "format": "mcq_lettered", "question_col": "question", "answer_col": "answer",
         "sample_cap": None, "strip_reasoning": False, "hf_config": None},
        {"repo": "ChuGyouk/AI_healthcare_QA", "data_type": "medical_knowledge",
         "format": "qa_pair", "question_col": "question", "answer_col": "gpt4o",
         "sample_cap": 5000, "strip_reasoning": False, "hf_config": None},
        {"repo": "ChuGyouk/HealthSearchQA-ko", "data_type": "medical_knowledge",
         "format": "qa_pair", "question_col": "question_ko", "answer_col": "answer_ko",
         "sample_cap": None, "strip_reasoning": False, "hf_config": None},
        {"repo": "hcw0329/medical-korean-alpaca", "data_type": "medical_knowledge",
         "format": "qa_pair", "question_col": "instruction", "answer_col": "output",
         "sample_cap": 5000, "strip_reasoning": False, "hf_config": None},

        {"repo": "ChuGyouk/medical-o1-reasoning-SFT-Ko", "data_type": "medical_reasoning",
         "format": "qa_pair", "question_col": "Question", "answer_col": "Response",
         "sample_cap": 5000, "strip_reasoning": True, "hf_config": None},
        {"repo": "ChuGyouk/ChainOfDiagnosis-Ko", "data_type": "medical_reasoning",
         "format": "dialogue", "question_col": None, "answer_col": None,
         "sample_cap": 5000, "strip_reasoning": True, "hf_config": None},
        {"repo": "ChuGyouk/MedQA-Evol-Korean", "data_type": "medical_reasoning",
         "format": "qa_pair", "question_col": "input", "answer_col": "output",
         "sample_cap": 5000, "strip_reasoning": False, "hf_config": None},

        {"repo": "squarelike/ko_medical_chat", "data_type": "conversation_style",
         "format": "dialogue", "question_col": None, "answer_col": None,
         "sample_cap": None, "strip_reasoning": False, "hf_config": None},
    ],

    # 9번 품질 필터링 임계값
    "min_answer_len": 5,
    "max_answer_len": 4000,

    # 10번 — 지금은 강제 오버샘플링 없음(비율만 보고함). 나중에 그룹별 비율을 강제하고
    # 싶으면 이 값을 채워서 10번 셀에서 사용하면 된다.
    "dataset_ratios": None,  # 예: {"medical_knowledge": 0.5, "medical_reasoning": 0.3, "conversation_style": 0.2}

    # 11번 Train/Val/Test 분리
    "train_ratio": 0.8,
    "val_ratio": 0.1,
    "test_ratio": 0.1,

    # 14번 QLoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": None,  # None이면 12번 로드 후 자동 탐지

    # 15~16번 학습
    "num_train_epochs": 2,
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "max_length": 1024,
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "logging_steps": 10,
    "eval_steps": 50,
    "save_steps": 50,
    "save_total_limit": 3,
    "early_stopping_patience": 3,

    # 20번 저장 — "-v2"는 이미 스크리닝 노트북(train_medgemma_lora.ipynb)의 KorMedMCQA 단독
    # 어댑터가 쓰고 있는 이름이라 충돌한다. 이 노트북은 "-main-v2"(구조화 파이프라인)로 구분.
    "adapter_repo": "gon-0130/medgemma-4b-lora-consultation-main-v2",
    "mixed_dataset_repo": "gon-0130/medgemma-mixed-dataset-main-v2",
    "colab_drive_path": "/content/drive/MyDrive/medgemma-lora-main-v2",
}

print(f"모델: {CONFIG['model_name']}")
print(f"데이터셋 {len(CONFIG['dataset_config'])}개 등록됨")
for d in CONFIG["dataset_config"]:
    cap = d["sample_cap"] if d["sample_cap"] is not None else "전량"
    print(f"  - [{d['data_type']:>18}] {d['repo']:<40} format={d['format']:<10} cap={cap}")


---

# CELL 4. Hugging Face 로그인

MedGemma 접근 승인 + Colab Secrets에 `HF_TOKEN` 등록이 먼저 필요하다
(왼쪽 메뉴 열쇠 아이콘 → Secrets → `HF_TOKEN`).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN이 없습니다. Colab Secrets에 HF_TOKEN을 등록하세요.")

login(token=HF_TOKEN)
print("Hugging Face 로그인 완료")


---

# CELL 5. 데이터셋 로드

`CONFIG["dataset_config"]`에 등록된 순서대로 전부 로드해서 `raw_datasets` 딕셔너리(키: repo id)에
담아둔다. **`sample_cap`이 걸려있는 데이터셋은 여기서 바로 셔플 후 상한만큼 잘라낸다** — 8번
(컬럼 매핑/변환)까지 전체 원본을 그대로 들고 있지 않도록, 로드 시점에 범위를 확정한다.
`sample_cap=None`인 데이터셋(ClinicalQA/HealthSearchQA-ko/ko_medical_chat)은 전량 유지.

In [ ]:
from datasets import load_dataset

raw_datasets = {}
for d in CONFIG["dataset_config"]:
    repo = d["repo"]
    try:
        # 일부 데이터셋(MedQA 등)은 language/version별 config 이름을 명시해야 로드된다.
        if d.get("hf_config"):
            ds = load_dataset(repo, d["hf_config"], split="train")
        else:
            ds = load_dataset(repo, split="train")
        original_n = len(ds)
        if d["sample_cap"]:
            ds = ds.shuffle(seed=CONFIG["seed"]).select(range(min(d["sample_cap"], original_n)))
        raw_datasets[repo] = ds
        cap_note = f"(원본 {original_n}개 중 상한 적용)" if d["sample_cap"] and original_n > d["sample_cap"] else ""
        print(f"[OK] {repo}: {len(ds)}개 {cap_note}")
    except Exception as e:
        print(f"[FAIL] {repo}: {type(e).__name__}: {e}")

print(f"\n로드 성공: {len(raw_datasets)}/{len(CONFIG['dataset_config'])}개")
missing = [d["repo"] for d in CONFIG["dataset_config"] if d["repo"] not in raw_datasets]
if missing:
    print("로드 실패한 데이터셋(진행 전에 원인 확인 필요):", missing)


---

# CELL 6. 데이터셋 구조 및 샘플 분석

각 데이터셋의 컬럼 목록과 첫 샘플을 출력한다 — `CONFIG`에 적어둔 `question_col`/`answer_col`이
실제 컬럼명과 맞는지 여기서 눈으로 확인하고, 다르면 3번 CONFIG로 돌아가 고친다.

In [ ]:
for d in CONFIG["dataset_config"]:
    repo = d["repo"]
    if repo not in raw_datasets:
        continue
    ds = raw_datasets[repo]
    cap_label = f"(상한 {d['sample_cap']} 적용됨)" if d["sample_cap"] else "(전량)"
    print("=" * 90)
    print(f"{repo}  (data_type={d['data_type']}, format={d['format']})")
    print(f"  로드된 개수: {len(ds)} {cap_label}")
    print(f"  컬럼: {ds.column_names}")
    if d["format"] == "qa_pair":
        for col, label in ((d["question_col"], "question_col"), (d["answer_col"], "answer_col")):
            status = "OK" if col in ds.column_names else "!! 컬럼 없음 — CONFIG 수정 필요"
            print(f"  {label}='{col}' -> {status}")
    print(f"  샘플[0]: {ds[0]}")


---

# CELL 7. 공통 전처리 함수

8번(컬럼 매핑/변환)과 9번(품질 검증)에서 공용으로 쓰는 정제 유틸리티. 숫자·의료 특수문자
(`℃ % / + - ( ) : , .`)·영어 의학 용어는 건드리지 않고, 다음만 처리한다:

- 공백 정규화, zero-width 문자 제거, 제어문자 제거, Unicode 정규화(NFC)
- 전화번호/이메일 패턴 마스킹(과탐 방지를 위해 패턴을 좁게 잡음)
- reasoning 태그(`<think>`/`<analysis>`/`Reasoning:`/`Step N:`) 스트리핑 — 내부 추론 과정을
  사용자에게 그대로 노출하지 않기 위함(단, 태그 뒤에 남는 실제 설명 텍스트는 보존)
- 반복 문자만 있는 저품질 텍스트 판별

In [ ]:
import re
import unicodedata

_ZERO_WIDTH_RE = re.compile(r"[\u200b\u200c\u200d\ufeff]")
_CONTROL_CHAR_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_WS_RE = re.compile(r"\s+")

_PHONE_RE = re.compile(r"(01[016789]|02|0[3-6][1-4])[-. ]?\d{3,4}[-. ]?\d{4}")
_EMAIL_RE = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")

# <think>...</think>, <analysis>...</analysis> 블록 전체와, "Reasoning:"/"Step 1:" 같은
# 라인 단위 프리픽스를 제거한다. 태그 안의 문장 자체가 최종 설명으로 이어지는 경우가 있어
# 태그/프리픽스만 지우고 뒤따르는 텍스트는 남긴다.
_REASONING_BLOCK_RE = re.compile(r"<think>.*?</think>|<analysis>.*?</analysis>", re.DOTALL | re.IGNORECASE)
_REASONING_LINE_PREFIX_RE = re.compile(r"^(reasoning|step\s*\d+)\s*[:：]\s*", re.IGNORECASE | re.MULTILINE)


def normalize_ws(text):
    if text is None:
        return ""
    return _WS_RE.sub(" ", str(text)).strip()


def clean_text(text, strip_reasoning=False):
    """zero-width/제어문자 제거, Unicode 정규화, (옵션)reasoning 태그 스트리핑, 공백 정규화."""
    if text is None:
        return ""
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = _ZERO_WIDTH_RE.sub("", text)
    text = _CONTROL_CHAR_RE.sub("", text)
    if strip_reasoning:
        text = _REASONING_BLOCK_RE.sub("", text)
        text = _REASONING_LINE_PREFIX_RE.sub("", text)
    return normalize_ws(text)


def mask_pii(text):
    """전화번호/이메일로 보이는 패턴을 마스킹하고 (마스킹된 텍스트, 마스킹 횟수)를 반환."""
    if not text:
        return text, 0
    count = 0

    def _sub(pattern, repl, s):
        nonlocal count
        s, n = pattern.subn(repl, s)
        count += n
        return s

    text = _sub(_EMAIL_RE, "[이메일]", text)
    text = _sub(_PHONE_RE, "[전화번호]", text)
    return text, count


_REPEAT_CHAR_RE = re.compile(r"(.)\1{9,}")  # 같은 글자가 10번 이상 반복


def is_low_quality(text, min_len, max_len):
    """반복문자뿐이거나 길이가 임계값 밖이면 (True, 사유)를 반환."""
    if not text:
        return True, "empty"
    if len(text) < min_len:
        return True, "too_short"
    if len(text) > max_len:
        return True, "too_long"
    stripped = _REPEAT_CHAR_RE.sub("", text)
    if len(stripped) < len(text) * 0.3:
        return True, "repeated_chars"
    return False, None


print("전처리 유틸리티 정의 완료: normalize_ws / clean_text / mask_pii / is_low_quality")


---

# CELL 8. 데이터셋별 컬럼 매핑 및 변환

`format`(`qa_pair`/`mcq_abcd`/`dialogue`)에 따라 서로 다른 변환 함수를 태우고, 전부
`{"messages": [...], "source_dataset": repo, "data_type": ...}` 공통 포맷으로 만든다.

- `qa_pair`: `question_col`/`answer_col` 한 쌍을 user/assistant 메시지로
- `mcq_abcd`: MedQA류 A~D 4지선다 — 선택지를 질문에 포함시키고 정답 문장을 완성형으로 생성
- `dialogue`: 멀티턴 대화 리스트(`conversations` 등) — 마지막 턴이 assistant인 경우만 채택

CELL 7의 `clean_text`/`mask_pii`를 여기서 적용하고, `strip_reasoning=True`인 데이터셋
(medical-o1, ChainOfDiagnosis-Ko)은 `<think>`/`Reasoning:` 류 태그를 여기서 제거한다.
(`sample_cap` 적용은 5번 로드 시점에서 이미 끝났으므로 여기서는 다시 자르지 않는다.)

In [ ]:
from datasets import Dataset

MCQ_OPTION_KEYS = ["A", "B", "C", "D"]


def _row_qa_pair(example, d):
    q = clean_text(example.get(d["question_col"]))
    a = clean_text(example.get(d["answer_col"]), strip_reasoning=d["strip_reasoning"])
    if not q or not a:
        return None
    q, _ = mask_pii(q)
    a, _ = mask_pii(a)
    return [{"role": "user", "content": q}, {"role": "assistant", "content": a}]


def _row_mcq_abcd(example, d):
    options_text = "\n".join(
        f"{key}. {clean_text(example[key])}"
        for key in MCQ_OPTION_KEYS
        if example.get(key) is not None and clean_text(example[key])
    )
    question = clean_text(example.get(d["question_col"]))
    if not question or not options_text:
        return None
    answer_text = clean_text(example.get(d["answer_col"]))
    try:
        letter = MCQ_OPTION_KEYS[int(example.get("answer_idx"))]
    except (TypeError, ValueError, IndexError):
        letter = None
    model_turn = f"정답은 {letter}번, {answer_text}입니다." if letter else f"정답은 {answer_text}입니다."
    user_turn = f"{question}\n\n선택지:\n{options_text}"
    return [{"role": "user", "content": user_turn}, {"role": "assistant", "content": model_turn}]


def _row_mcq_lettered(example, d):
    """options가 {"option_A": ..., "option_B": ...} 딕셔너리이고 답 컬럼이 "C"처럼 글자
    하나뿐인 데이터용(snuh/ClinicalQA). explanation이 있으면 같이 붙인다 — 정답 글자만
    쓰면 답변이 1글자라 품질 필터에 걸려 전부 삭제되는 문제가 있었다."""
    question = clean_text(example.get(d["question_col"]))
    options = example.get("options")
    if not question or not isinstance(options, dict):
        return None
    options_text = "\n".join(
        f"{key.replace('option_', '').upper()}. {clean_text(val)}"
        for key, val in sorted(options.items())
        if val is not None and clean_text(val)
    )
    if not options_text:
        return None
    answer_letter = str(example.get(d["answer_col"], "")).strip().upper()
    answer_text = clean_text(options.get(f"option_{answer_letter}", ""))
    if not answer_letter or not answer_text:
        return None
    explanation = clean_text(example.get("explanation"))
    user_turn = f"다음 의학 문제에 답하세요.\n\n질문:\n{question}\n\n선택지:\n{options_text}"
    model_turn = f"정답은 {answer_letter}입니다.\n\n{answer_letter}. {answer_text}"
    if explanation:
        model_turn += f"\n\n해설: {explanation}"
    user_turn, _ = mask_pii(user_turn)
    model_turn, _ = mask_pii(model_turn)
    return [{"role": "user", "content": user_turn}, {"role": "assistant", "content": model_turn}]


def _extract_dialogue_turns(example):
    for key in ("conversations", "CoD_conversations", "dialogue", "conversation"):
        value = example.get(key)
        if value:
            return value
    return None


def _row_dialogue(example, d):
    turns = _extract_dialogue_turns(example)
    if not turns or len(turns) < 2:
        return None

    def _role(turn):
        raw = str(turn.get("from") or turn.get("role") or "").lower()
        return "assistant" if raw in ("doctor", "assistant", "gpt") else "user"

    def _text(turn):
        raw = turn.get("value") or turn.get("content") or ""
        cleaned = clean_text(raw, strip_reasoning=d["strip_reasoning"])
        cleaned, _ = mask_pii(cleaned)
        return cleaned

    raw_messages = [{"role": _role(t), "content": _text(t)} for t in turns if _text(t)]

    # 원본 데이터에 같은 화자(doctor 등)의 턴이 연속으로 나오는 경우가 있다(긴 답변이 길이
    # 제한으로 쪼개진 흔적으로 보임) — Gemma의 chat template은 user/assistant가 반드시
    # 번갈아 나와야 해서(TemplateError: roles must alternate), 연속된 같은 역할 턴은 하나로
    # 합쳐서 교대 구조를 만든다.
    messages = []
    for m in raw_messages:
        if messages and messages[-1]["role"] == m["role"]:
            messages[-1]["content"] = (messages[-1]["content"] + " " + m["content"]).strip()
        else:
            messages.append(dict(m))

    if len(messages) < 2 or messages[0]["role"] != "user" or messages[-1]["role"] != "assistant":
        return None
    return messages


_FORMAT_HANDLERS = {
    "qa_pair": _row_qa_pair, "mcq_abcd": _row_mcq_abcd,
    "mcq_lettered": _row_mcq_lettered, "dialogue": _row_dialogue,
}

converted_rows = []
removal_log = {}

for d in CONFIG["dataset_config"]:
    repo = d["repo"]
    if repo not in raw_datasets:
        continue
    rows = raw_datasets[repo]  # 5번에서 이미 sample_cap이 적용된 상태
    handler = _FORMAT_HANDLERS[d["format"]]

    total = len(rows)
    kept = dropped_invalid = dropped_quality = 0

    for example in rows:
        messages = handler(example, d)
        if messages is None:
            dropped_invalid += 1
            continue
        answer_text = messages[-1]["content"]
        bad, _reason = is_low_quality(answer_text, CONFIG["min_answer_len"], CONFIG["max_answer_len"])
        if bad:
            dropped_quality += 1
            continue
        converted_rows.append({"messages": messages, "source_dataset": repo, "data_type": d["data_type"]})
        kept += 1

    removal_log[repo] = {"total": total, "kept": kept, "dropped_invalid": dropped_invalid, "dropped_quality": dropped_quality}
    print(f"{repo:<45} {total:>6} -> {kept:>6}   (invalid={dropped_invalid}, quality={dropped_quality})")

print(f"\n총 변환 완료: {len(converted_rows)}개")


---

# CELL 9. 데이터 품질 검증

전처리 전/후 개수, 제거 사유별 개수, 완전 중복 제거 결과, 질문/답변 길이 통계, 데이터셋별
최종 개수, 랜덤 샘플 10개를 출력한다 — "정확도"뿐 아니라 "이 데이터가 실제로 믿을 만한가"를
학습 전에 먼저 눈으로 확인하기 위한 단계.

In [ ]:
import random
from collections import Counter

total_before = sum(v["total"] for v in removal_log.values())
total_invalid = sum(v["dropped_invalid"] for v in removal_log.values())
total_quality = sum(v["dropped_quality"] for v in removal_log.values())

# 완전히 동일한 question+answer 쌍 제거
seen = set()
deduped_rows = []
dup_count = 0
for row in converted_rows:
    key = (row["messages"][0]["content"], row["messages"][-1]["content"])
    if key in seen:
        dup_count += 1
        continue
    seen.add(key)
    deduped_rows.append(row)

converted_rows = deduped_rows

q_lens = [len(r["messages"][0]["content"]) for r in converted_rows]
a_lens = [len(r["messages"][-1]["content"]) for r in converted_rows]

print("=" * 60)
print("데이터 품질 검증 결과")
print("=" * 60)
print(f"원본 총합      : {total_before}")
print(f"형식 오류 제거  : {total_invalid}")
print(f"품질 필터 제거  : {total_quality}")
print(f"완전 중복 제거  : {dup_count}")
print(f"최종 남은 개수  : {len(converted_rows)}")
print()
print(f"질문 평균 길이  : {sum(q_lens)/len(q_lens):.1f}  (min={min(q_lens)}, max={max(q_lens)})")
print(f"답변 평균 길이  : {sum(a_lens)/len(a_lens):.1f}  (min={min(a_lens)}, max={max(a_lens)})")
print()
print("데이터셋별 최종 개수:")
for repo, n in Counter(r["source_dataset"] for r in converted_rows).items():
    print(f"  {repo:<45} {n}")

print("\n랜덤 샘플 10개:")
random.seed(CONFIG["seed"])
for row in random.sample(converted_rows, min(10, len(converted_rows))):
    print("-" * 60)
    print(f"[{row['source_dataset']} / {row['data_type']}]")
    print("Q:", row["messages"][0]["content"][:150])
    print("A:", row["messages"][-1]["content"][:150])


---

# CELL 10. 데이터셋 병합 및 비율 확인

정제된 행을 하나의 `Dataset`으로 합치고, `data_type`별 비율을 "있는 그대로" 보고한다.
초기에는 강제 오버샘플링을 하지 않는다 — `CONFIG["dataset_ratios"]`를 채우면 이후 그룹별
강제 비율 조정 로직을 추가할 수 있는 자리만 마련해둔다.

In [ ]:
dataset = Dataset.from_list(converted_rows)
print(f"병합 완료: 총 {len(dataset)}개\n")

type_counts = Counter(dataset["data_type"])
print("data_type별 개수/비율 (원본 비율 그대로, 강제 오버샘플링 없음):")
for dtype, n in type_counts.items():
    print(f"  {dtype:<20} {n:>6}개  ({n/len(dataset):.1%})")

if CONFIG["dataset_ratios"]:
    print("\n주의: CONFIG['dataset_ratios']가 설정되어 있지만 이 셀은 아직 강제 리샘플링을 "
          "적용하지 않습니다 — 필요해지면 이 값을 사용해 그룹별로 다시 샘플링하는 로직을 추가하세요.")


---

# CELL 11. Train / Validation / Test 분리

정규화된 질문 텍스트 기준으로 한 번 더 중복을 제거한 뒤(같은 질문이 다른 데이터셋에 다른
답변으로 실려 있으면 Train/Test에 나뉘어 들어가는 데이터 누수 위험이 있음) `train_ratio`
/`val_ratio`/`test_ratio`(기본 80/10/10)로 분리한다. `SEED` 고정으로 재현 가능하게 한다.

In [ ]:
seen_q = set()
keep_idx = []
for i, row in enumerate(dataset):
    q_key = normalize_ws(row["messages"][0]["content"]).lower()
    if q_key in seen_q:
        continue
    seen_q.add(q_key)
    keep_idx.append(i)

before_n = len(dataset)
dataset = dataset.select(keep_idx)
print(f"질문 기준 추가 중복 제거: {before_n} -> {len(dataset)}개 ({before_n - len(dataset)}개 제거)")

split1 = dataset.train_test_split(test_size=CONFIG["val_ratio"] + CONFIG["test_ratio"], seed=CONFIG["seed"])
train_dataset = split1["train"]
rest = split1["test"]

rel_test_ratio = CONFIG["test_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"])
split2 = rest.train_test_split(test_size=rel_test_ratio, seed=CONFIG["seed"])
val_dataset = split2["train"]
test_dataset = split2["test"]

print(f"Train: {len(train_dataset)}개 ({len(train_dataset)/len(dataset):.1%})")
print(f"Val  : {len(val_dataset)}개 ({len(val_dataset)/len(dataset):.1%})")
print(f"Test : {len(test_dataset)}개 ({len(test_dataset)/len(dataset):.1%})")


---

# CELL 12. 모델 및 Tokenizer 로드

1번에서 확인한 GPU 세대에 따라 4bit 양자화의 `compute_dtype`을 bf16(Ampere+)/fp16(T4 등)으로
자동 선택한다. `MODEL_NAME`만 바꾸면 Gemma/MedGemma/Qwen 등 다른 모델로도 그대로 재사용
가능하지만, 모델마다 chat template/특수토큰이 달라 13번에서 별도로 확인한다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

_is_ampere_plus = torch.cuda.get_device_capability(0)[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if _is_ampere_plus else torch.float16
USE_BF16 = _is_ampere_plus

print("GPU 세대 기준 정밀도 자동 선택:", "bf16 (Ampere+)" if USE_BF16 else "fp16 (T4 등, AMP는 사용 안 함)")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
    token=HF_TOKEN,
)

print("모델/토크나이저 로드 완료:", CONFIG["model_name"])


---

# CELL 13. Chat Template 적용

`messages`를 trl의 `SFTConfig(completion_only_loss=True)`가 요구하는 prompt/completion
포맷(마지막 assistant 턴만 completion, 나머지는 prompt)으로 나눈다. `tokenizer.apply_chat_template()`
으로 실제 렌더링 결과를 확인하고, chat_template이 없는 모델을 쓰는 경우를 대비한 fallback도
함께 정의해둔다.

In [ ]:
def messages_to_prompt_completion(example):
    msgs = example["messages"]
    return {"prompt": msgs[:-1], "completion": [msgs[-1]]}


has_chat_template = getattr(tokenizer, "chat_template", None) is not None
print("tokenizer.chat_template 존재:", has_chat_template)


def render_fallback(messages):
    return "\n".join(f"<{m['role']}>\n{m['content']}\n</{m['role']}>" for m in messages)


train_dataset = train_dataset.map(messages_to_prompt_completion)
val_dataset = val_dataset.map(messages_to_prompt_completion)
test_dataset = test_dataset.map(messages_to_prompt_completion)

sample = train_dataset[0]
if has_chat_template:
    rendered = tokenizer.apply_chat_template(sample["prompt"] + sample["completion"], tokenize=False)
else:
    rendered = render_fallback(sample["prompt"] + sample["completion"])
    print("경고: 이 모델은 chat_template이 없습니다 — fallback 포맷을 사용합니다. "
          "학습 전에 프롬프트 구조가 모델에 맞는지 직접 검증하세요.")

print("렌더링 결과:\n")
print(rendered)


---

# CELL 14. QLoRA 설정

`target_modules`를 `CONFIG`에서 직접 지정하지 않았으면(`None`), 모델 구조에서 attention/MLP
projection 계층(`q_proj`/`k_proj`/`v_proj`/`o_proj`/`gate_proj`/`up_proj`/`down_proj`)을
자동 탐지한다 — 모델을 Gemma/Qwen 등으로 바꿔도 레이어 이름 차이에 안전하게 대응하기 위함.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

_PROJ_NAME_RE = re.compile(r"(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$")


def detect_target_modules(model):
    names = set()
    for name, module in model.named_modules():
        if module.__class__.__name__ in ("Linear4bit", "Linear8bitLt") and _PROJ_NAME_RE.search(name):
            names.add(name.split(".")[-1])
    return sorted(names) or ["q_proj", "k_proj", "v_proj", "o_proj"]


target_modules = CONFIG["target_modules"] or detect_target_modules(model)
print("target_modules:", target_modules)

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


---

# CELL 15. Trainer 설정

Google Drive를 마운트해 체크포인트를 세션 밖에 남긴다(마운트 실패 시 `/content` 로컬 경로로
대체). Validation loss 기준 **Early Stopping**(`early_stopping_patience`)을 적용하려면
`eval_strategy`/`save_strategy`가 같은 주기(`steps`)로 맞춰져 있어야 하고
`load_best_model_at_end=True`가 필요하다 — 그래야 학습이 멈춘 시점이 아니라 **가장 좋았던
체크포인트**가 최종 모델로 남는다.

**trl 버전 방어**: `SFTConfig`가 받는 인자 이름이 trl 버전마다 바뀐 이력이 있다
(`max_seq_length`→`max_length`, `warmup_ratio` 지원 여부 등). 원하는 값을 딕셔너리로 만든
뒤, 지금 설치된 `SFTConfig`가 실제로 받는 인자만 걸러서 넘긴다 — 지원 안 하는 인자는 에러
대신 경고만 찍고 건너뛴다.

In [ ]:
import os
import inspect
from google.colab import drive
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

try:
    drive.mount("/content/drive")
    output_dir = CONFIG["colab_drive_path"]
except Exception as e:
    print("Drive 마운트 실패, 로컬 경로로 대체합니다:", e)
    output_dir = "/content/medgemma-lora-v2"
os.makedirs(output_dir, exist_ok=True)
print("체크포인트 저장 경로:", output_dir)

desired_sft_config = dict(
    output_dir=output_dir,
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    num_train_epochs=CONFIG["num_train_epochs"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    bf16=USE_BF16,
    fp16=False,  # AMP(fp16=True)는 Gemma LoRA 레이어 일부(bfloat16) + GradScaler 충돌 이력 있음 — 항상 끔
    gradient_checkpointing=True,
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    completion_only_loss=True,
    max_length=CONFIG["max_length"],
    report_to="none",
    packing=False,
)

_sft_accepted_params = set(inspect.signature(SFTConfig.__init__).parameters)
_sft_dropped = {k: v for k, v in desired_sft_config.items() if k not in _sft_accepted_params}
if _sft_dropped:
    print(f"[WARNING] 설치된 trl의 SFTConfig가 지원하지 않아 무시된 인자: {list(_sft_dropped.keys())}")

training_args = SFTConfig(**{k: v for k, v in desired_sft_config.items() if k in _sft_accepted_params})

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["early_stopping_patience"])],
)

print(trainer)


---

# CELL 16. Fine-tuning 실행

학습 시작 전 모델/GPU/데이터 개수/파라미터 수를 먼저 출력해서 잘못된 설정으로 몇 시간을
날리는 걸 방지한다.

In [ ]:
print("=" * 60)
print("학습 시작 전 정보")
print("=" * 60)
print("모델:", CONFIG["model_name"])
print("GPU:", torch.cuda.get_device_name(0))
print("GPU 메모리(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("Train 데이터 개수:", len(train_dataset))
print("Validation 데이터 개수:", len(val_dataset))

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체 파라미터 수: {total_params:,}")
print(f"학습 가능한 파라미터 수: {trainable_params:,} ({trainable_params/total_params:.2%})")

trainer.train()


---

# CELL 16-1. (선택) 세션이 끊긴 경우 체크포인트에서 재개

Colab 세션이 끊겼다가 다시 연결됐다면, 바로 위 CELL 16(`trainer.train()`)을 처음부터 다시
누르지 말고 **이 셀을 대신 실행**한다. `output_dir`(Drive)에 남아있는 가장 최근 체크포인트를
찾아서 그 지점부터 이어서 학습한다. 체크포인트가 없으면 안내만 하고 끝낸다 — 이 경우 CELL 16
으로 가서 처음부터 시작하면 된다.

**주의**: 세션이 끊겼다는 건 커널이 초기화됐다는 뜻이라, 이 셀 이전에 CELL 1~15(모델 로드,
LoRA 설정, Trainer 생성까지)는 다시 실행해야 `trainer`/`model` 변수가 존재한다. 단순 브라우저
탭 새로고침은 커널을 안 끊으므로 이 셀이 필요 없다 — 원래 CELL 16이 이어서 잘 돌아간다.

In [ ]:
checkpoint_dirs = [
    os.path.join(output_dir, name)
    for name in os.listdir(output_dir)
    if name.startswith("checkpoint-") and os.path.isdir(os.path.join(output_dir, name))
]

if checkpoint_dirs:
    checkpoint_dirs.sort(key=lambda p: int(os.path.basename(p).split("-")[-1]))
    latest_checkpoint = checkpoint_dirs[-1]
    print("최신 체크포인트에서 재개:", latest_checkpoint)
    trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("체크포인트가 없습니다 — 위 CELL 16에서 처음부터 학습을 시작하세요.")


---

# CELL 17. 학습 결과 및 Loss 확인

`trainer.state.log_history`에서 step별 train loss / eval loss / learning rate를 뽑아서
표로 확인하고, validation loss가 가장 낮았던 지점(Early Stopping이 최종적으로 되돌아간
지점과 같아야 함)을 출력한다.

In [ ]:
import pandas as pd

log_df = pd.DataFrame(trainer.state.log_history)
display_cols = [c for c in ["step", "loss", "eval_loss", "learning_rate"] if c in log_df.columns]
print(log_df[display_cols].dropna(how="all", subset=[c for c in ["loss", "eval_loss"] if c in display_cols]).to_string(index=False))

if "eval_loss" in log_df.columns:
    eval_losses = log_df.dropna(subset=["eval_loss"])[["step", "eval_loss"]]
    if len(eval_losses):
        best = eval_losses.loc[eval_losses["eval_loss"].idxmin()]
        print(f"\n최소 validation loss: {best['eval_loss']:.4f} (step {int(best['step'])})")


---

# CELL 18. Test 평가

Test Loss를 확인한다. 의료 모델이므로 Loss 하나만으로 성능을 판단하지 않는다는 원칙에 따라,
나중에 별도의 의료 QA 평가셋(예: PubMedQA-test-Ko, MedQA test)을 연결할 수 있는 확장 포인트
(`evaluate_medical_qa`)만 마련해두고, 실제 품질 판단은 19번의 사람 검토를 기본으로 삼는다.

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
print("Test 평가 결과:", test_metrics)


def evaluate_medical_qa(trainer, dataset, extra_scorer=None):
    """향후 별도 의료 QA 평가셋을 연결할 수 있는 확장 포인트.
    extra_scorer(question, gold_answer, generated_answer) -> dict 콜백을 넘기면 Test Loss
    외의 커스텀 지표(정확도/관련성 등)를 계산하도록 확장할 수 있다. 자동 점수만으로 의료적
    정확성을 확정하지 않기 위해, 실제 판단은 19번의 사람 검토를 기본 절차로 둔다."""
    metrics = trainer.evaluate(eval_dataset=dataset, metric_key_prefix="test")
    if extra_scorer is None:
        return metrics
    return metrics


print("\n(별도 의료 QA 평가셋이 준비되면 evaluate_medical_qa()의 extra_scorer를 채워 확장)")


---

# CELL 19. Base Model vs Fine-tuned Model 비교

Test set에서 무작위로 뽑은 질문에 대해 **같은 베이스 모델**로 `model.disable_adapter()`를 써서
LoRA 어댑터를 끈 답변(base)과 켠 답변(fine-tuned)을 나란히 비교한다 — 별도로 base 모델을 다시
로드할 필요 없이, 지금 메모리에 있는 모델 그대로 어댑터만 잠깐 끄고 켜는 방식이라 빠르다.

In [ ]:
import random as _random

model.eval()
model.config.use_cache = True
model.gradient_checkpointing_disable()

_random.seed(CONFIG["seed"])
sample_indices = _random.sample(range(len(test_dataset)), min(8, len(test_dataset)))


def generate_answer(model, messages, max_new_tokens=200):
    if has_chat_template:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = render_fallback(messages) + "\n<assistant>\n"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=0.7, top_p=0.9, repetition_penalty=1.3, no_repeat_ngram_size=3,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


for idx in sample_indices:
    row = test_dataset[idx]
    question = row["prompt"][-1]["content"] if row["prompt"] else ""
    print("=" * 90)
    print("질문:", question[:200])

    ft_answer = generate_answer(model, row["prompt"])
    print("\n[Fine-tuned]", ft_answer[:400])

    with model.disable_adapter():
        base_answer = generate_answer(model, row["prompt"])
    print("\n[Base(어댑터 비활성화)]", base_answer[:400])


---

# CELL 20. LoRA Adapter 및 결과 저장

1) LoRA Adapter + Tokenizer, 2) Training Config, 3) 전처리된 통합 Dataset(HF Hub),
4) Train/Val/Test 개수 정보, 5) 학습 로그를 각각 저장하고, 팀 공유용으로 Adapter를
Hugging Face Hub(private)에도 올린다.

In [ ]:
import json as _json

FINAL_ADAPTER_DIR = os.path.join(output_dir, "final_adapter")

# 1. LoRA Adapter + Tokenizer
trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print("Adapter 저장:", FINAL_ADAPTER_DIR)

# 2. Training Config
config_path = os.path.join(output_dir, "training_config.json")
with open(config_path, "w", encoding="utf-8") as f:
    _json.dump(CONFIG, f, ensure_ascii=False, indent=2, default=str)
print("Config 저장:", config_path)

# 3. 전처리된 통합 Dataset
dataset.push_to_hub(CONFIG["mixed_dataset_repo"], private=True)
print(f"통합 데이터셋 업로드: https://huggingface.co/datasets/{CONFIG['mixed_dataset_repo']}")

# 4. Train/Val/Test 정보
split_info = {"train": len(train_dataset), "val": len(val_dataset), "test": len(test_dataset), "seed": CONFIG["seed"]}
split_info_path = os.path.join(output_dir, "split_info.json")
with open(split_info_path, "w", encoding="utf-8") as f:
    _json.dump(split_info, f, ensure_ascii=False, indent=2)
print("Split 정보 저장:", split_info_path)

# 5. 학습 로그
log_path = os.path.join(output_dir, "training_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    _json.dump(trainer.state.log_history, f, ensure_ascii=False, indent=2, default=str)
print("학습 로그 저장:", log_path)

# 6. Hugging Face Hub에 Adapter 업로드 (팀 공유용)
model.push_to_hub(CONFIG["adapter_repo"], private=True)
tokenizer.push_to_hub(CONFIG["adapter_repo"], private=True)
print(f"Adapter 업로드: https://huggingface.co/{CONFIG['adapter_repo']}")

print("\n=== 전체 파이프라인 완료 ===")
